# Analysis of RQ1 data

How well can LLM match human annotator on entire dataset?

## making data the format we expect

In [62]:
# constants

ALL_FOUNDATIONS  = {'Authority', 'Purity', 'Care', 'Thin Morality', 'Non-Moral'}
EXCLUSIVE_LABELS = {'Thin Morality', 'Non-Moral'}
FOUNDATION_COLS  = list(ALL_FOUNDATIONS)
LLM_ANNOTATOR    = 'LLM'

In [53]:
import pandas as pd

df_llm = pd.read_csv('./data_nlpforsocialinteractions/MFRC_llama_annotations_all.csv')
df_humans = pd.read_csv('./data_nlpforsocialinteractions/MFRC_annotator_based_nlpforsocialinteractions.csv', sep='\t')
df_human_agg = pd.read_csv('./data_nlpforsocialinteractions/MFRC_nlpforsocialinteractions.csv', sep='\t')

In [54]:
# df_llm has fewer entries since it comes from the aggregated df, suggesting some texts dropped during agg in original dataset.

df_llm["text"].nunique()

17457

In [55]:
df_humans["text"].nunique()

17886

In [56]:
df_human_agg["text"].nunique()

17457

In [57]:
df_humans['item_id'] = df_humans['text'].apply(lambda x: hash(str(x).strip()))
df_human_agg['item_id'] = df_human_agg['text'].apply(lambda x: hash(str(x).strip()))
df_llm['item_id'] = df_llm['text'].apply(lambda x: hash(str(x).strip()))


In [99]:
human_ids = set(df_human_agg['item_id'].unique())
llm_ids   = set(df_llm['item_id'].unique())
shared_ids         = human_ids & llm_ids
human_only_ids     = human_ids - llm_ids
llm_only_ids       = llm_ids - human_ids

df_humans_only = df_humans[df_humans['item_id'].isin(human_only_ids)]
df_llm_only    = df_llm[df_llm['item_id'].isin(llm_only_ids)]

print(f"Items in humans only (dropped): {len(human_only_ids)}")
print(f"Items in LLM only    (dropped): {len(llm_only_ids)}")
print(f"Shared items (kept):            {len(shared_ids)}")


Items in humans only (dropped): 0
Items in LLM only    (dropped): 0
Shared items (kept):            17457


In [100]:
# violation check
# violations are for e.g. when llm annotates 'care' and 'thin morality' for the same item, which is against instructions (thin morality should be exclusive)
# do not apply for the human_df, because each row is aggregated

violations = []

def validate_and_build(label_set, annotator, item_id, check_exclusivity=True):
    if check_exclusivity:
        exclusive_present = label_set & EXCLUSIVE_LABELS
        if exclusive_present and len(label_set) > 1:
            violations.append({
                'item_id': item_id, 'annotator': annotator,
                'label_set': str(label_set), 'type': 'label_conflict'
            })
            return None
    return label_set


In [116]:
human_data = []

foundation_cols = ['Authority', 'Purity', 'Care', 'Thin Morality', 'Non-Moral']

def agg_to_frozenset(row, cols):
    # total annotators for this item = sum of all counts
    n_annotators = sum(row[f] for f in cols)
    if n_annotators == 0:
        return frozenset()  # no annotations at all
    threshold = n_annotators / 2
    return frozenset(f for f in cols if row[f] > threshold)


for _, row in df_human_agg.iterrows():
    label_set = agg_to_frozenset(row, foundation_cols)
    label_set = validate_and_build(label_set, 'human_aggregated', row['item_id'], check_exclusivity=False)
    if label_set is not None:
        human_data.append(('human_aggregated', row['item_id'], label_set))


In [117]:
llm_data = []
foundation_cols = ['Authority', 'Purity', 'Care', 'Thin Morality', 'Non-Moral']

for _, row in df_llm.iterrows():
    label_set = frozenset(f for f in foundation_cols if row[f] == 1)
    label_set = validate_and_build(label_set, LLM_ANNOTATOR, row['item_id'])
    if label_set is not None:
        llm_data.append((LLM_ANNOTATOR, row['item_id'], label_set))

In [118]:
violations

[{'item_id': 5162410832240550048,
  'annotator': 'LLM',
  'label_set': "frozenset({'Non-Moral', 'Authority'})",
  'type': 'label_conflict'},
 {'item_id': 5162410832240550048,
  'annotator': 'LLM',
  'label_set': "frozenset({'Non-Moral', 'Authority'})",
  'type': 'label_conflict'},
 {'item_id': 5162410832240550048,
  'annotator': 'LLM',
  'label_set': "frozenset({'Non-Moral', 'Authority'})",
  'type': 'label_conflict'}]

In [119]:
## report violations
# lol looks like some of these humans don't obey instructions.
# most often thin morality is labelled along with other moralities (when it should be an exclusive label)

print(f"Violations skipped: {len(violations)}")
if violations:
    vdf = pd.DataFrame(violations)
    print(vdf['annotator'].value_counts().to_string())
    vdf.to_csv('violations.csv', index=False)

Violations skipped: 3
annotator
LLM    3


In [120]:
all_data = human_data + llm_data

In [121]:
all_data

[('human_aggregated', -8799435231411789585, frozenset()),
 ('human_aggregated', -6349358376756382276, frozenset({'Non-Moral'})),
 ('human_aggregated', 6222460520199581611, frozenset({'Authority'})),
 ('human_aggregated', -8266370570183014687, frozenset()),
 ('human_aggregated', -1971835582121193275, frozenset({'Non-Moral'})),
 ('human_aggregated', -802372219609878399, frozenset()),
 ('human_aggregated', 3030627273809872437, frozenset()),
 ('human_aggregated', 2880360323998148935, frozenset()),
 ('human_aggregated', -7624446931940775143, frozenset()),
 ('human_aggregated', 3938990335098947726, frozenset({'Non-Moral'})),
 ('human_aggregated', -3883594095826753979, frozenset({'Non-Moral'})),
 ('human_aggregated', -1910671270066683756, frozenset({'Care'})),
 ('human_aggregated', -7805226404836950437, frozenset()),
 ('human_aggregated', -1412877824137382339, frozenset({'Non-Moral'})),
 ('human_aggregated', 4126490518190005468, frozenset()),
 ('human_aggregated', -4777995013650675526, frozen

In [122]:
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics import binary_distance, masi_distance

# because the humans labelled out of scope columns (e.g. authority) we need to assign a sentinel label
# llm did not return any labels for some cols too!
human_map = {item_id: ls for _, item_id, ls in human_data}
llm_map   = {item_id: ls for _, item_id, ls in llm_data}

valid_ids = {i for i in human_map if i in llm_map 
             and len(human_map[i]) > 0 
             and len(llm_map[i]) > 0}

n_empty_human = sum(1 for i in human_map if len(human_map[i]) == 0)
n_empty_llm   = sum(1 for i in llm_map   if len(llm_map[i])   == 0)
n_both_empty  = sum(1 for i in human_map if i in llm_map 
                    and len(human_map[i]) == 0 and len(llm_map[i]) == 0)

print(f"Empty human frozensets (excluded foundations only): {n_empty_human}")
print(f"Empty LLM frozensets:                               {n_empty_llm}")
print(f"Items dropped (at least one side empty):            {len(human_map) - len(valid_ids)}")
print(f"Items kept for α computation:                       {len(valid_ids)}")


def add_sentinel(label_set):
    return label_set if label_set else frozenset({'Other'})

valid_ids = set(human_map.keys()) & set(llm_map.keys())

filtered_data = (
    [('human_aggregated', i, add_sentinel(human_map[i])) for i in valid_ids] +
    [(LLM_ANNOTATOR,      i, add_sentinel(llm_map[i]))   for i in valid_ids]
)


# Overall α (MASI)
task_overall = AnnotationTask(data=filtered_data, distance=masi_distance)
print(f"\nOverall α (MASI): {task_overall.alpha():.4f}")

# Per-foundation α (nominal binary)
print(f"\n{'Foundation':<20} {'α (nominal)':>12}")
print("-" * 34)
for f in sorted(foundation_cols):
    binary_data = []
    for annotator, item_id, label_set in filtered_data:
        binary_label = f if f in label_set else f'NOT_{f}'
        binary_data.append((annotator, item_id, binary_label))
    
    task_f = AnnotationTask(data=binary_data, distance=binary_distance)
    print(f"{f:<20} {task_f.alpha():>12.4f}")


Empty human frozensets (excluded foundations only): 4082
Empty LLM frozensets:                               502
Items dropped (at least one side empty):            4494
Items kept for α computation:                       12963

Overall α (MASI): 0.0591

Foundation            α (nominal)
----------------------------------
Authority                 -0.1448
Care                       0.2104
Non-Moral                  0.2234
Purity                    -0.0932
Thin Morality             -0.0426


## Results and interpretation

### Strict Majority

When using agreement with strict majority of human annotators (> n/2)

```
Empty human frozensets (excluded foundations only): 4082
Empty LLM frozensets:                               502
Items dropped (at least one side empty):            4494
Items kept for α computation:                       12963

Overall α (MASI): 0.0591

Foundation            α (nominal)
----------------------------------
Authority                 -0.1448
Care                       0.2104
Non-Moral                  0.2234
Purity                    -0.0932
Thin Morality             -0.0426
```

overall a is just above 0, which means that llm only does slightly better than chance

per foundation
- best is non-moral, which is the easiest
- next best is care, which is somewhat more universal
- thin morality is around chance level
- purity, authority kind of were less than that

### Lenient majority (>= n/2)



```
Empty human frozensets (excluded foundations only): 1963
Empty LLM frozensets:                               502
Items dropped (at least one side empty):            2419
Items kept for α computation:                       15038

Overall α (MASI): 0.0783

Foundation            α (nominal)
----------------------------------
Authority                 -0.0470
Care                       0.3123
Non-Moral                  0.1322
Purity                    -0.0290
Thin Morality             -0.0707
```

The foundations that changed here had exactly 2/4 annotators annotating them (high human disagreemtn).
- care changed the most (positive) from 0.2104 to 0.3123, which is reasonably good llm alignment
- non moral dropped: llm disagrees with borderline cases where exactly 2 humans thought it was nonmoral
- purity, thin morality, authority still show systemic diagreement between llm and human

### confidence comparison

LLM had better confidence on easy to rate categories (care, nonmoral)
But was systematically overconfident on rarer categories (esp thin morality: agreement was 0)

In [129]:
from tabulate import tabulate

llm_conf_map = dict(zip(df_llm['item_id'], df_llm['confidence']))

CONFIDENCE_WEIGHTS = {
    'Very Confident':     1.0,
    'Somewhat Confident': 0.5,
    'Not Confident':      0.25
}

rows = []
for f in sorted(foundation_cols):
    agree_conf    = [CONFIDENCE_WEIGHTS.get(llm_conf_map[i], 0)
                     for i in valid_ids if i in human_map and i in llm_map
                     and f in human_map[i] and f in llm_map[i]]
    disagree_conf = [CONFIDENCE_WEIGHTS.get(llm_conf_map[i], 0)
                     for i in valid_ids if i in human_map and i in llm_map
                     and (f in human_map[i]) != (f in llm_map[i])]

    rows.append([
        f,
        f"{sum(agree_conf)/len(agree_conf):.3f}"    if agree_conf    else "n/a",
        f"{sum(disagree_conf)/len(disagree_conf):.3f}" if disagree_conf else "n/a",
        len(agree_conf),
        len(disagree_conf)
    ])

print(tabulate(rows,
               headers=['Foundation', 'Agree conf', 'Disagree conf', 'N agree', 'N disagree'],
               tablefmt='rounded_outline'))


╭───────────────┬──────────────┬─────────────────┬───────────┬──────────────╮
│ Foundation    │ Agree conf   │   Disagree conf │   N agree │   N disagree │
├───────────────┼──────────────┼─────────────────┼───────────┼──────────────┤
│ Authority     │ 0.811        │           0.712 │       481 │         7096 │
│ Care          │ 0.903        │           0.751 │      1257 │         4358 │
│ Non-Moral     │ 1.000        │           0.707 │      3739 │         6511 │
│ Purity        │ 0.846        │           0.719 │       133 │         4496 │
│ Thin Morality │ n/a          │           0.751 │         0 │         1427 │
╰───────────────┴──────────────┴─────────────────┴───────────┴──────────────╯
